# 6.1 Dimensionaliteitsreductie, en wat het met je doet

Elke dataset tot nu toe had weinig genoeg kolommen om direct te plotten. Deze niet: een handgeschreven cijfer bestaat uit 784 getallen, en er bestaat geen scatterplot van 784 dimensies. Twee methoden drukken dat plat tot een plaatje waar je naar kunt kijken.

Beide werken. Beide bevatten ook een beslissing die ze niet voor jou gaan nemen, en beide leveren hoe dan ook een zelfverzekerd ogend plaatje op:

- **PCA** zoekt de richtingen waarlangs de data het meest varieert — gemeten in de eenheden die jij hebt aangeleverd. Geef het grammen en het vindt grammen.
- **t-SNE** rangschikt punten zodat buren buren blijven. Hoeveel buren is een parameter, en de vorm van het antwoord hangt daarvan af.

Dit notebook bouwt op vanaf een schets die je met het blote oog kunt controleren — wat "de richting van de meeste variantie" betekent, en waarom je daarmee het minste verliest — naar de twee showcases waar de methoden de mist in gaan. De secties met de titel "waar het misgaat" zijn bewust opgezet: het falen is opzettelijk en de juiste versie staat er direct naast.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from goad_toolkit.visualizer import PlotSettings, ProjectionPlot, ScreePlot
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.datasets import fetch_openml, make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from wa_analyzer.data import load_showcase

rng = np.random.default_rng(42)


## 6.1.1 Waar PCA naar zoekt

Drie kolommen reduceren naar twee betekent een plat vlak kiezen en elk punt daarop laten vallen; twee reduceren naar één betekent een *lijn* kiezen. Begin bij de lijn, want daar zie je het hele idee in één schets.

Neem een puntenwolk waarvan het gemiddelde is afgetrokken, en een willekeurige lijn door de oorsprong. Elk punt valt uiteen in twee delen: waar het landt *langs* de lijn, en hoe ver het *van* de lijn af ligt. Volgens Pythagoras geldt: `(afstand tot oorsprong)² = (positie langs de lijn)² + (afstand tot de lijn)²`, en de linkerkant hangt niet af van welke lijn je hebt gekozen. Dus gesommeerd over de hele wolk geldt: **de lijn met de meeste spreiding erlangs is precies de lijn waar de punten het dichtst bij liggen.**

Dat is heel PCA in één zin: de richting van de grootste variantie is ook de projectie die het minste verliest. De schets tekent beide helften — een meetkundig plaatje, geen grafiek van data, dus gewoon standaard matplotlib.

Rood is wat de projectie weggooit; de zwarte stippen zijn wat behouden blijft. Roteer de lijn en de rode lijnstukken groeien precies even snel als de zwarte spreiding krimpt.

In [ ]:
cloud = rng.multivariate_normal([0, 0], [[3.0, 2.2], [2.2, 2.0]], size=40)
cloud -= cloud.mean(axis=0)

direction = np.linalg.svd(cloud)[2][0]  # the line with the most spread along it
along = cloud @ direction  # every point's position along that line
feet = np.outer(along, direction)  # where every point lands on it

fig, ax = plt.subplots(figsize=(6, 6))
reach = np.abs(along).max() * 1.1
ax.plot([-reach * direction[0], reach * direction[0]], [-reach * direction[1], reach * direction[1]],
        color="black", linewidth=1, label="the line PCA picks")
for point, foot in zip(cloud, feet):
    ax.plot([point[0], foot[0]], [point[1], foot[1]], color="crimson", linewidth=0.8, alpha=0.7)
ax.scatter(cloud[:, 0], cloud[:, 1], color="#4c72b0", zorder=3, label="data, centred")
ax.scatter(feet[:, 0], feet[:, 1], color="black", s=14, zorder=4, label="the same points, projected")
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc="upper left")
_ = ax.set_title("Most spread along the line = least distance to the line")


### Drie kolommen die er eigenlijk twee zijn

Nu hetzelfde idee één dimensie hoger, op data die zo is geconstrueerd dat het antwoord bekend is. Honderd punten op een gekromde strook: twee kolommen volgen de strook, en de derde is een gewogen som van de eerste twee plus een beetje ruis. Drie getallen per rij, maar de rijen liggen allemaal dicht bij één gekanteld vlak — er zou dus een goed tweedimensionaal plaatje moeten bestaan, en PCA zou dat moeten vinden.

In [ ]:
def tilted_sheet(rng: np.random.Generator, m: int = 100, noise: float = 0.1) -> np.ndarray:
    """m points on a curved strip; the third column is 0.1 * first + 0.3 * second, plus noise."""
    angles = rng.random(m) * 3 * np.pi / 2 - 0.5
    sheet = np.empty((m, 3))
    sheet[:, 0] = np.cos(angles) + np.sin(angles) / 2 + noise * rng.standard_normal(m) / 2
    sheet[:, 1] = np.sin(angles) * 0.7 + noise * rng.standard_normal(m) / 2
    sheet[:, 2] = sheet[:, 0] * 0.1 + sheet[:, 1] * 0.3 + noise * rng.standard_normal(m)
    return sheet


sheet = tilted_sheet(np.random.default_rng(4))


### De drie delen van een SVD

`np.linalg.svd` op de gecentreerde data geeft drie onderdelen terug, en samen vormen ze precies de matrix waar je mee begon:

$$X = U \Sigma V^T$$

- De rijen van **$V^T$** zijn de richtingen — eerst de lijn uit de schets, dan de beste lijn loodrecht daarop, en dan degene loodrecht op beide. Drie kolommen erin, drie richtingen eruit.
- **$\Sigma$** bevat één getal per richting, gesorteerd van groot naar klein. Gekwadrateerd is elk getal de spreiding die de data langs die richting heeft, dus `s² / sum(s²)` is het aandeel van de variantie dat elke richting draagt.
- **$U$** geeft aan waar elke rij landt langs elke richting — de "positie langs de lijn" uit de schets, eenmaal per richting.

Drie woorden beschrijven wat de rijen van $V^T$ bruikbaar maakt als assen. Ze zijn **orthogonaal**: loodrecht op elkaar, dus wat de ene richting meet kan de volgende niet nog eens meten. **Genormaliseerd**: elk heeft lengte één, dus een coördinaat erlangs staat in de eigen eenheden van de data. En samen vormen ze een **basis**: elk punt in de data kan worden geschreven als zoveel van de eerste richting plus zoveel van de tweede plus zoveel van de derde, en er gaat niets verloren totdat je er eentje weglaat. Het weglaten van de laatste is de projectie.

In [ ]:
sheet_centred = sheet - sheet.mean(axis=0)
U, s, Vt = np.linalg.svd(sheet_centred)
U.shape, s.shape, Vt.shape


De data en de drie richtingen, waarbij de lengte van elke pijl zijn singuliere waarde toont ten opzichte van de grootste:

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(sheet_centred[:, 0], sheet_centred[:, 1], sheet_centred[:, 2])

for i, colour in enumerate(["r", "g", "b"]):
    direction = Vt[i] * s[i] / s[0]  # arrow length: this direction's share of the spread
    ax.quiver(0, 0, 0, *direction, color=colour, arrow_length_ratio=0.1, linewidths=3)

ax.set_xlabel("column 1")
ax.set_ylabel("column 2")
ax.set_zlabel("column 3")  # ty: ignore[unresolved-attribute] -- Axes3D, not the base Axes the stub sees
_ = ax.set_title("Three directions, arrow length = relative singular value")


De rode en groene pijlen liggen in het gekantelde vlak waar de punten op liggen; de blauwe is bijna onzichtbaar, omdat die recht uit dat vlak wijst en de enige spreiding in die richting de ruis is. Geen van de drie is een kolom van de data — de rode pijl is een mengsel van alle drie de kolommen — en dat is wat bedoeld wordt met "PCA vindt nieuwe assen".

Behoud de eerste twee richtingen en laat de derde vallen: `sheet_centred @ Vt.T[:, :2]` is de positie van elk punt langs de rode en groene pijlen, en niets anders.

In [ ]:
W2 = Vt.T[:, :2]
sheet_2d_svd = sheet_centred @ W2

settings = PlotSettings(
    figsize=(6, 5),
    title="The sheet, projected onto its first two directions",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=sheet_2d_svd, color="#4c72b0")


De gekromde strook, plat van voren gezien: het plaatje dat data ter grootte van een plat vlak verdient. Dat is heel PCA, en `sklearn`'s `PCA` doet exact dit — centreren, SVD, de eerste `n_components` richtingen behouden:

In [ ]:
pca = PCA(n_components=2)
sheet_2d = pca.fit_transform(sheet)
np.allclose(sheet_2d_svd, sheet_2d) or np.allclose(sheet_2d_svd, -sheet_2d)


Dezelfde coördinaten, op een minteken na: een richting en zijn spiegelbeeld beschrijven dezelfde lijn, dus een as in een PCA-plot kan tussen twee runs of twee bibliotheken gespiegeld uitvallen. Het plaatje betekent in beide gevallen hetzelfde.

`explained_variance_ratio_` is de `s² / sum(s²)` van hierboven, en `ScreePlot` is het plaatje daarvan. Lees dit vóórdat je enige projectie leest: het vertelt van hoeveel procent van de data de projectie een afbeelding is.

In [ ]:
pca = PCA().fit(sheet)
print("explained variance ratio per component:")
print(np.round(pca.explained_variance_ratio_, 4))
print("\nfrom the singular values, squared and normalised:")
print(np.round(np.square(s) / np.sum(np.square(s)), 4))

settings = PlotSettings(
    figsize=(6, 4),
    title=f"Two components carry {pca.explained_variance_ratio_[:2].sum():.1%} of the variance",
    xlabel="component",
    ylabel="share of variance",
)
fig, ax = ScreePlot(settings).plot(explained_variance_ratio=pca.explained_variance_ratio_)


## 6.1.2 Waar PCA misgaat: de eenheden die je eraan gaf

PCA maximaliseert variantie. Variantie heeft eenheden — gekwadrateerde — dus de component die het vindt is deels een feit over de data en deels een feit over waarin jij de dingen hebt gemeten. Op de strook hierboven was dat onzichtbaar, omdat alle drie de kolommen dezelfde verzonnen schaal hadden.

De pinguïns uit les 2 hebben dat niet. Bekijk de standaarddeviaties voordat je iets anders doet.

In [ ]:
penguins = load_showcase("penguins").dropna()
FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print(penguins[FEATURES].describe().loc[["count", "mean", "std"]].round(1).to_string())

In [ ]:
measurements = penguins[FEATURES].to_numpy()
unscaled = PCA().fit(measurements)

print(f"explained variance ratio: {np.round(unscaled.explained_variance_ratio_, 4)}")
print("\nwhat the first component is made of:")
print(pd.Series(unscaled.components_[0], index=FEATURES).round(3).to_string())

**De eerste component verklaart 99,99% van de variantie en dat is lichaamsgewicht (body mass)**, met een loading van 1,000 tegenover vrijwel nul voor al het andere. PCA faalde niet — het deed precies wat het belooft. Lichaamsgewicht heeft een standaarddeviatie van 805, snaveldiepte (bill depth) heeft 2,0, dus vrijwel alle variantie in deze matrix bestaat uit grammen, en de richting van de grootste variantie is de grammen-as.

Een getal van 99,99% in een rapport lijkt een triomf. Het is het symptoom.

De duidelijkste manier om te zien dat dit over eenheden gaat en niet over pinguïns: deel lichaamsgewicht door duizend en noem het kilogrammen. Dezelfde vogels, dezelfde metingen, dezelfde informatie.

In [ ]:
kilos = measurements.copy()
kilos[:, FEATURES.index("body_mass_g")] /= 1000
in_kilos = PCA().fit(kilos)

print(f"explained variance ratio: {np.round(in_kilos.explained_variance_ratio_, 4)}")
print("\nwhat the first component is made of now:")
print(pd.Series(in_kilos.components_[0], index=FEATURES).round(3).to_string())

De eerste component is nu **vleugellengte (flipper length)**, met een loading van 0,96, en verklaart 91,9% in plaats van 99,99%. Er is niets aan de vogels veranderd. Een eenheid wel.

Zet dus vóór PCA elke kolom op gelijke voet: trek het gemiddelde af en deel door de standaarddeviatie, en dat is wat `StandardScaler` doet. Daarna is één "eenheid" van elke kolom gelijk aan één standaarddeviatie van die kolom, en kan geen enkele kolom winnen door in iets kleins gemeten te zijn.

In [ ]:
standardised = StandardScaler().fit_transform(measurements)
scaled = PCA().fit(standardised)

print(f"explained variance ratio: {np.round(scaled.explained_variance_ratio_, 4)}\n")
loadings = pd.DataFrame(scaled.components_[:2].T, index=pd.Index(FEATURES), columns=pd.Index(["PC1", "PC2"]))
print(loadings.round(2).to_string())

Nu betekenen de componenten iets wat je hardop kunt uitspreken.

**PC1 is grootte**: snavellengte, vleugellengte en lichaamsgewicht laden allemaal positief, dus een pinguïn met een hoge score op PC1 is een grote pinguïn. Snavel*diepte* laadt negatief — wat de bevinding uit 5.2 is die via een andere weg terugkeert, omdat de soort met lange snavels juist ondiepe snavels heeft.

**PC2 is snavelvorm**: lengte en diepte samen, vleugel en gewicht op nul. Het is de as die aangeeft "hoe lomp/dik is de snavel", en het is een echt tweede kenmerk van een pinguïn in plaats van een afrondingsfout op het eerste.

Twee componenten, 88% van de variantie, en een zin per as. Vergelijk dat met de 99,99%-versie, die maar één as had en dat was een weegschaal.

In [ ]:
def separation(scores: np.ndarray) -> float:
    """Between-species variance over within-species variance, along one component."""
    frame = pd.DataFrame({"score": scores, "species": penguins.species.to_numpy()})
    return frame.groupby("species").score.mean().var() / frame.groupby("species").score.var().mean()


print(f"species separation along PC1, unscaled: "
      f"{separation(unscaled.transform(measurements)[:, 0]):.2f}")
print(f"species separation along PC1, scaled:   "
      f"{separation(scaled.transform(standardised)[:, 0]):.2f}")

settings = PlotSettings(
    figsize=(11, 4.5),  # ty: ignore[invalid-argument-type]
    title="The same 333 penguins, before and after scaling",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[
        f"raw units — PC1+PC2 = {unscaled.explained_variance_ratio_[:2].sum():.1%}",
        f"standardised — PC1+PC2 = {scaled.explained_variance_ratio_[:2].sum():.1%}",
    ],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=2)
for ax, model, values in [(axes[0], unscaled, measurements), (axes[1], scaled, standardised)]:
    host.plot_on_axes(ProjectionPlot(settings), ax,
                      coordinates=model.transform(values)[:, :2],
                      labels=penguins.species.to_numpy(), legend=(ax is axes[1]))

Het is niet alleen een kwestie van interpretatie: de ongeschaalde versie is **slechter in zijn taak**. Langs PC1 stijgt de verhouding tussen tussen-soorten- en binnen-soortenvariantie van 3,09 naar 9,50 zodra de kolommen zijn gestandaardiseerd. Links trekt Ezelspinguïn (Gentoo) weg — dat zijn de zware vogels — maar Adelie en Chinstrap blijven grondig door elkaar gemengd, omdat het enige dat hen scheidt snavelvorm is en snavelvorm in millimeters gemeten wordt. Rechts komen alle drie netjes los.

En kijk naar wat het linkerpaneel je niet vertelt. De twee assen zijn 99,99% en 0,01% van de variantie, dus eerlijk getekend zou het een horizontale lijn zijn; matplotlib heeft de tweede as uitgerekt om het paneel te vullen, waardoor een component van 0,01% nu op de helft van het plaatje lijkt. **Een projectieplot vult altijd zijn kader.** Dat is precies waarom de scree plot niet optioneel is.

> **De regel.** Standaardiseer vóór PCA, tenzij elke kolom al dezelfde eenheid heeft *én* het je bedoeling is dat kolommen met een grotere spreiding zwaarder meetellen. Pixel-intensiteiten (allemaal 0–255) zijn de gebruikelijke uitzondering — en dat is waarom de MNIST-sectie hieronder niet schaalt.

## 6.1.3 De swiss roll: een schaduw, of een kaart van buurten

Schalen fixt wat PCA meet; het verandert niet wat PCA *is*, namelijk een platte schaduw. De swiss roll is de standaard dataset om de grens daarvan te zien: een tweedimensionaal vel, opgerold in drie dimensies. Kleur is de positie langs het vel — het enige dat een plaatje van deze data zou moeten behouden.

In [ ]:
roll, colour = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(roll[:, 0], roll[:, 1], roll[:, 2], c=colour, cmap=plt.cm.hot)  # ty: ignore[unresolved-attribute]
ax.view_init(10, -70)  # ty: ignore[unresolved-attribute] -- Axes3D, not the base Axes the stub sees
_ = ax.set_title("A two-dimensional sheet, rolled up in three dimensions")


Eerst PCA. Het behoudt 71% van de variantie, wat klinkt als het grootste deel van de data, en het plaatje is een spiraal gezien vanaf de zijkant.

In [ ]:
pca = PCA(n_components=2)
roll_2d = pca.fit_transform(roll)

settings = PlotSettings(
    figsize=(6, 5),
    title=f"Swiss roll under PCA — {pca.explained_variance_ratio_.sum():.0%} of the variance",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=roll_2d, labels=colour, palette="hot",
                                        legend=False)

Kijk naar waar de kleuren terechtkomen. Geel is het ene uiteinde van het vel en zwart is het andere, zover mogelijk uit elkaar langs het vel — en in de schaduw zitten ze vlak naast elkaar, omdat de windingen van de rol vanuit elk plat gezichtspunt overlappen. De schaduw is trouw aan de 3D-geometrie, maar nutteloos voor de vraag "welke punten liggen dicht bij elkaar *op het vel*".

t-SNE stelt die vraag direct. In plaats van een gezichtspunt zoekt het naar een 2D-opstelling waarin de dichtstbijzijnde buren van elk punt in 3D nog steeds zijn dichtstbijzijnde buren zijn.

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
roll_tsne = tsne.fit_transform(roll)

settings = PlotSettings(
    figsize=(6, 5),
    title="Swiss roll under t-SNE",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=roll_tsne, labels=colour, palette="hot",
                                        legend=False)


De kleur loopt nu van het ene uiteinde naar het andere: zwart aan de ene kant, geel aan de andere kant, het vel min of meer uitgerold. Dat is wat PCA niet kon, omdat geen enkel plat gezichtspunt een kromming ongedaan maakt. Kijk echter wat het heeft gekost. Het vel is verscheurd in klonten met gaten ertussen, en die gaten zitten niet in de data — de rol is één doorlopende strook. t-SNE heeft de buren van elk punt behouden en al het andere opgegeven: hoe ver twee klonten uit elkaar liggen, en of een gat echt is, zijn dingen die dit plaatje je niet kan vertellen.

Dat is de afruil in één zin: **PCA behoudt de globale geometrie en kan een kromming niet ongedaan maken; t-SNE ontrolt de kromming en geeft de globale geometrie op.**

### De manifold-hypothese

De rol is een tweedimensionaal ding dat toevallig in drie dimensies leeft. De aanname achter het gebruik van t-SNE op echte data is dat hetzelfde geldt op grote schaal: dat 784 pixelwaarden van een handgeschreven cijfer zich niet echt over 784 onafhankelijke richtingen verspreiden, maar op een gekromd, veel lager-dimensionaal oppervlak binnen die ruimte liggen — een *manifold* — omdat een "7" maar op een beperkt aantal manieren kan variëren. Als dat klopt, zou een plaatje dat het oppervlak uitrolt tien cijfers als tien groepen moeten tonen, terwijl een platte schaduw dat niet zou moeten kunnen. Controleer het op de cijfers zelf.

## 6.1.4 MNIST: 784 pixels

De download betreft 70.000 afbeeldingen en gaat eenmalig naar `~/.cache/mads-dav`.

In [ ]:
cache = Path.home() / ".cache/mads-dav"
cache.mkdir(parents=True, exist_ok=True)

mnist = fetch_openml("mnist_784", version=1, as_frame=False, data_home=str(cache))
mnist.target = mnist.target.astype(np.uint8)


In [ ]:
digits = mnist["data"]
digit_labels = mnist["target"]
print(f"{digits.shape[0]:,} digits, each one {digits.shape[1]} pixels")

Zeventigduizend cijfers is meer dan t-SNE aankan: het rekent ongeveer kwadratisch in het aantal punten, en dit notebook fit het zes keer. Vijfduizend, willekeurig getrokken, is genoeg om alles hieronder te zien en verandert een koffiepauze in een paar seconden.

Dat is ook een beslissing. Schrijf het op wanneer je het doet — een plaatje van een steekproef is een plaatje van een steekproef.

In [ ]:
rng = np.random.default_rng(42)
sample = rng.choice(len(digits), size=5000, replace=False)
subset, subset_labels = digits[sample], digit_labels[sample]
print(f"{subset.shape[0]:,} digits, {subset.shape[1]} pixels each")

Elk cijfer is een 28x28 afbeelding platgeslagen tot een rij van 784 getallen, één per pixel. Hier is de eerste, weer teruggevouwen tot een vierkant:

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(digits[0].reshape(28, 28), cmap="gray")
ax.set_xticks([])
ax.set_yticks([])
_ = ax.set_title(f"digit {digit_labels[0]}, as 784 numbers")


Een rij van 784 pixel-intensiteiten, allemaal op dezelfde schaal van 0–255 — wat het ene geval is waarin niet-standaardiseren vóór PCA een verdedigbare keuze is: elke kolom deelt al dezelfde eenheid, en een pixel die sterker varieert tussen cijfers *moet* zwaarder meetellen. Eerst de schaduw, met zijn scree plot ernaast.

In [ ]:
pca = PCA().fit(subset)
cumulative = np.cumsum(pca.explained_variance_ratio_)

print(f"the first two components carry {cumulative[1]:.1%} of the variance")
for target in (0.5, 0.8, 0.95):
    print(f"  components needed for {target:.0%}: {int(np.argmax(cumulative >= target)) + 1}")

settings = PlotSettings(
    figsize=(7, 4),
    title="784 pixels, and how the variance is spread over them",
    xlabel="component",
    ylabel="share of variance",
)
fig, ax = ScreePlot(settings).plot(explained_variance_ratio=pca.explained_variance_ratio_,
                                   n_components=20)

In [ ]:
settings = PlotSettings(
    figsize=(7, 6),
    title=f"MNIST under PCA — two components, {cumulative[1]:.1%} of the variance",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=pca.transform(subset)[:, :2],
                                        labels=subset_labels, palette="tab10", s=8)

De cijfers overlappen tot één grote vlek, en de scree plot legt uit waarom: **twee componenten dragen slechts 17,0% van de variantie**, en je hebt er 148 nodig voor 95%. Het plaatje is niet fout; het is een afbeelding van 17% van deze dataset, en dat getal hoort in de titel — en dat is waar `ScreePlot` en de titel erboven voor zijn bedoeld.

Dit is de eerlijke versie van "PCA doet iets, maar het is niet erg nuttig in twee dimensies". Het is niet nutteloos; het toont je de 17% die past.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, init="pca")
mnist_tsne = tsne.fit_transform(subset)

settings = PlotSettings(
    figsize=(7, 6),
    title="MNIST under t-SNE, default perplexity of 30",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=mnist_tsne, labels=subset_labels,
                                        palette="tab10", s=8)

Tien klonten, één per cijfer, en de paren die elkaar raken zijn de paren die op elkaar lijken. Dit is t-SNE die echt werk verricht dat PCA niet kon: de manifold is gekromd, en PCA kan er alleen met een plat vlak doorheen snijden. De aanname uit §6.1.3 heeft zich uitbetaald.

Nu de parameter.

## 6.1.5 Waar t-SNE misgaat: perplexity is geen weergave-optie

Perplexity is ruwweg "hoeveel buren elk punt moet proberen dichtbij te houden". Het heeft een standaardwaarde van 30, en standaarden zijn waar beslissingen zich verstoppen. Dezelfde 5.000 cijfers, vier waarden.

In [ ]:
PERPLEXITIES = (5, 30, 100, 200)
embeddings = {
    perplexity: TSNE(n_components=2, perplexity=perplexity, random_state=42,
                     init="pca").fit_transform(subset)
    for perplexity in PERPLEXITIES
}

settings = PlotSettings(
    figsize=(11, 9),
    title="One dataset, four perplexities",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[f"perplexity {perplexity}" for perplexity in PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(PERPLEXITIES))
for ax, perplexity in zip(axes, PERPLEXITIES):
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embeddings[perplexity],
                      labels=subset_labels, palette="tab10", legend=False, s=8)

In [ ]:
pairs = np.triu_indices(10, k=1)
gaps = {}
for perplexity, embedding in embeddings.items():
    frame = pd.DataFrame(embedding, columns=pd.Index(["x", "y"])).assign(digit=subset_labels)
    centroids = frame.groupby("digit")[["x", "y"]].mean().to_numpy()
    distances = np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=-1)
    gaps[perplexity] = distances[pairs] / distances[pairs].max()
    print(f"perplexity {perplexity:>3d}: silhouette by true digit "
          f"{silhouette_score(embedding, subset_labels):.2f}")

print("\nspearman between the pairwise gaps each perplexity draws:")
table = pd.DataFrame(
    [[spearmanr(gaps[a], gaps[b]).statistic for b in PERPLEXITIES] for a in PERPLEXITIES],
    index=pd.Index(PERPLEXITIES), columns=pd.Index(PERPLEXITIES),
)
print(table.round(2).to_string())

names = [f"{a}-{b}" for a in range(10) for b in range(a + 1, 10)]
print("\nwhich digits the picture puts closest together, and furthest apart:")
for perplexity in (PERPLEXITIES[0], PERPLEXITIES[-1]):
    order = np.argsort(gaps[perplexity])
    print(f"  perplexity {perplexity:>3d}: closest {[names[i] for i in order[:3]]}, "
          f"furthest {[names[i] for i in order[-3:]]}")


Twee verschillende antwoorden, en je hebt ze allebei nodig.

**Lidmaatschap is stabiel.** De silhouette-score afgezet tegen het echte cijfer blijft tussen 0,20 en 0,28 bij alle vier. Welke perplexity je ook kiest, de tien groepen komen eruit als tien groepen. Wanneer je zegt "hier zijn clusters", ondersteunt t-SNE je.

**Geometrie is dat niet.** De gaten tussen die clusters, gerangschikt, komen tussen perplexity 5 en 100 overeen met een Spearman-correlatie van slechts 0,41. Bij perplexity 5 zijn de twee cijfers die het verst uit elkaar liggen 0 en 1; bij perplexity 200 zijn het 0 en 7. Dus *hoe ver twee klonten uit elkaar liggen* is een feit over de parameter, niet over handschrift.

Het enige dat overeind blijft is 4-9 als het dichtstbijzijnde paar bij beide uitersten, wat de verwarring is die iedereen zou voorspellen. Een relatie die standhoudt over instellingen heen is het rapporteren waard. De rangschikking van één enkel plaatje is dat niet.

> **Wat je mag zeggen over een t-SNE-plot:** welke punten bij elkaar zijn beland, en of dat een verandering van perplexity overleeft. **Wat je niet mag zeggen:** dat één cluster dichter bij een tweede ligt dan bij een derde, dat een gat breed is, dat een cluster groot is, of wat dan ook over de assen — en dat is waarom `ProjectionPlot` de tick marks verbergt.

## 6.1.6 De fout die mensen papers kost

Alles hierboven bevatte echte structuur — tien cijfers zijn echt tien verschillende dingen. Kijk nu eens wat hetzelfde gereedschap doet met data die totaal géén structuur heeft: duizend punten, vijftig dimensies, elke waarde een onafhankelijke trekking uit dezelfde normale verdeling. Er zijn geen groepen. Er is niets te vinden.

In [ ]:
noise = np.random.default_rng(0).normal(size=(1000, 50))

NOISE_PERPLEXITIES = (2, 5, 30, 100)
settings = PlotSettings(
    figsize=(11, 9),
    title="Fifty dimensions of pure noise, four perplexities",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[f"perplexity {perplexity}" for perplexity in NOISE_PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(NOISE_PERPLEXITIES))

scores = {}
for ax, perplexity in zip(axes, NOISE_PERPLEXITIES):
    embedding = TSNE(n_components=2, perplexity=perplexity, random_state=0,
                     init="pca").fit_transform(noise)
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embedding, color="#777777")
    scores[perplexity] = silhouette_score(
        embedding, KMeans(5, n_init=10, random_state=0).fit_predict(embedding))


In [ ]:
raw_score = silhouette_score(noise, KMeans(5, n_init=10, random_state=0).fit_predict(noise))
print(f"k-means with k=5 on the raw 50-dimensional noise: silhouette {raw_score:.3f}")
for perplexity, score in scores.items():
    print(f"  ...on the t-SNE picture at perplexity {perplexity:>3d}: silhouette {score:.3f}")

Bij perplexity 2 breekt het plaatje zichtbaar op in kleine klontjes. Er zijn geen klonten. Het algoritme werd gevraagd om elk punt dicht bij zijn twee dichtstbijzijnde buren te houden, en in vijftig dimensies ruis heeft elk punt twee dichtstbijzijnde buren, dus het deed braaf zijn werk.

De getallen vertellen iets dat nog erger is, en dat geldt bij *elke* perplexity. Draai k-means op de ruwe data en het silhouette is **0,016** — waarmee het correct rapporteert dat er niets is. Draai exact dezelfde k-means op het tweedimensionale plaatje en het is rond de **0,31**, wat je in elke andere context zou interpreteren als een behoorlijke clusterstructuur.

**Cluster daarom nooit de embedding.** Afstanden in een t-SNE-plot zijn niet de afstanden van de data; de projectie heeft vijftig dimensies samengeperst op een begrensd stuk papier, en alles wat daar afstand meet, meet het papier. Cluster de data, en gebruik de projectie om te kijken naar wat je hebt gevonden.

## Wat je moet noteren

1. **Heb je geschaald, en waarom.** "Alle kolommen waren pixels" is een geldige reden. "Het werkte in beide gevallen" is dat niet.
2. **De variantie die je plaatje bevat** — uit de scree plot, in het bijschrift. Twee componenten van de 784 is een getal dat een lezer niet kan raden.
3. **De perplexity**, en één andere perplexity die je hebt bekeken. Als de bevinding alleen bestaat bij één instelling, is het een bevinding over de instelling.
4. **Welke claim je maakt**: dat punten samen clusteren, of waar die groepen liggen ten opzichte van elkaar. Het eerste kan standhouden. Het tweede niet.

---

**Waar dit naartoe gaat.** [06.2](06.2-modelling%20jeffrey.ipynb) behoudt de cijfers en stelt een andere vraag — niet "hoe ziet dit eruit", maar "kan een model ze uit elkaar houden", wat de controle is of een vorm die je in een projectie zag het waard was om in te geloven.